# Data preparation of hydro reservoir level filling and inflows derived from aggregation from ENTSO-E transparency

source: https://transparency.entsoe.eu/

Creates the following parsed datasets

- Hourly storage levels per country for selected year for setting storage start and end conditions (reservoir_level_'+year+'_hourly_entsoe_TP.csv) 
- Weekly storage levels per country for selected year for setting storage start and end conditions (reservoir_level_'+year+'_weekly_entsoe_TP.csv) 

Settings in next window

In [1]:
#download files again (yes/no)?
download = "no"

#set year for data creation
year = '2024'

In [2]:
import pysftp
import sys
import os
import pandas as pd
import numpy as np

c:\Users\jonas\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,
c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
#enter your password here
password = "3?mJ}V4us?!L}5E"                
#enter email address here
username = "jsavelsberg@ethz.ch"                
port = '22'

In [4]:
cnopts = pysftp.CnOpts()
cnopts.hostkeys = None

c:\Users\jonas\anaconda3\Lib\site-packages\pysftp\__init__.py:61: UserWarning: Failed to load HostKeys from C:\Users\jonas\.ssh\known_hosts.  You will need to explicitly load HostKeys (cnopts.hostkeys.load(filename)) or disableHostKey checking (cnopts.hostkeys = None).
  warnings.warn(wmsg, UserWarning)


In [5]:
dir_out = "../parsed_data/"

In [6]:
fn_additional = "../additional_data.xlsx"
df_countries= pd.read_excel(fn_additional, sheet_name='Countries_EU', index_col="Country")
countries = list(df_countries.index)

In [7]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/') 

In [9]:
if download=='yes':
    with pysftp.Connection(host=host, username=username, password=password, cnopts=cnopts) as sftp:
        print("Connection succesfully established.")

        # show list of files
        files = sftp.listdir('/TP_export/')   
        #print(files)

## load data

In [ ]:
#set paths and get file names
path_level = path+'AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D/'
path_level_local = path_local+'hydro_ENTSOE/reservoir_level_transparency/'
if download=='yes':
    with pysftp.Connection(host=host, username=username, password=password, cnopts=cnopts) as sftp:
        print("Connection succesfully established.")
        # show list of files
        files = sftp.listdir(path_level)
if download =="no": 
    files = os.listdir(path_level_local)        
if year != "":
    files = [i for i in files if year in i]
print(files)

In [12]:
#download reservoir level data (AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D)
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password, cnopts=cnopts) as sftp:
        for file in files:
            sftp.get(path_level+file,path_level_local+file)
            print('Successfully downloaded file '+file)

In [13]:
df_temp = pd.read_csv(path_level_local+"2024_01_AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D.csv", decimal=".",sep="\t",parse_dates=True, index_col="DateTime")
df_temp.head()

,ResolutionCode,AreaCode,AreaTypeCode,AreaName,MapCode,StoredEnergy,DeletedFlag,UpdateTime
DateTime,,,,,,,,
2024-01-01,P7D,10Y1001A1001A44P,BZN,SE1 BZN,SE1,7668000.0,0,2024-01-10 16:27:09.009
2024-01-08,P7D,10Y1001A1001A44P,BZN,SE1 BZN,SE1,7290000.0,0,2024-01-17 14:48:45.045
2024-01-15,P7D,10Y1001A1001A44P,BZN,SE1 BZN,SE1,6709000.0,0,2024-01-24 15:49:06.006
2024-01-22,P7D,10Y1001A1001A44P,BZN,SE1 BZN,SE1,6411000.0,0,2024-01-31 18:06:19.019
2024-01-29,P7D,10Y1001A1001A44P,BZN,SE1 BZN,SE1,6306000.0,0,2024-02-07 18:48:32.032


In [14]:
df_temp.ResolutionCode.unique()

array(['P7D'], dtype=object)

In [15]:
#combine files to one data frame
df_level = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_level_local+file,
                          decimal=".",sep="\t",
                          parse_dates=True, index_col="DateTime").drop("AreaCode", axis=1)
    df_level = pd.concat([df_level,df_temp])
df_level = df_level[df_level.AreaTypeCode == "CTY"].drop(["AreaTypeCode",'ResolutionCode','AreaName','UpdateTime','DeletedFlag'], axis=1).reset_index()
df_level = df_level.sort_values(by=['DateTime'])
df_level = df_level.rename(columns={"MapCode": "country", 'DateTime':'date','StoredEnergy':'MWh'})
#add timestamp for first hour for all countries for upsampling in next step
for country in countries:
    df_level = pd.concat([df_level, pd.DataFrame([[pd.Timestamp(str(np.int64(year)-1)+'-12-31'),country,float("NaN")]], columns=['date','country','MWh'])], ignore_index=True)
df_level.head()

,date,country,MWh
0,2024-01-01,GE,440.0
1,2024-01-01,RS,584000.0
2,2024-01-01,ES,9429185.0
3,2024-01-01,FI,3410560.0
4,2024-01-01,FR,2736874.0


In [16]:
#values are reported weekly so we have to resample to hourly values
#upsample with backwardsfill and select baseyear again
df_level_hourly = df_level.pivot(index='date',columns='country')[['MWh']].resample('H').fillna("bfill")
df_level_hourly = df_level_hourly[pd.DatetimeIndex(df_level_hourly.reset_index().date).year.astype(str) == year].stack()
df_level_hourly.tail()

C:\Users\jonas\AppData\Local\Temp\ipykernel_46860\979772684.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_level_hourly = df_level.pivot(index='date',columns='country')[['MWh']].resample('H').fillna("bfill")
C:\Users\jonas\AppData\Local\Temp\ipykernel_46860\979772684.py:3: FutureWarning: DatetimeIndexResampler.fillna is deprecated and will be removed in a future version. Use obj.ffill(), obj.bfill(), or obj.nearest() instead.
  df_level_hourly = df_level.pivot(index='date',columns='country')[['MWh']].resample('H').fillna("bfill")
C:\Users\jonas\AppData\Local\Temp\ipykernel_46860\979772684.py:4: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_level_hourly = df_level_hourly[pd.DatetimeIndex(df_level_hourly.reset_index

MWh
date       country            
2024-12-30 PT        1872486.0
           RO        1764600.0
           RS         460000.0
           SE       26123000.0
           SI           2600.0

In [17]:
#we also export weekly values
df_level = df_level.set_index(['date','country'])

In [18]:
df_level.to_csv(dir_out+'reservoir_level_'+year+'_weekly_entsoe_TP.csv', encoding="utf-8", index=True)
df_level_hourly.to_csv(dir_out+'reservoir_level_'+year+'_hourly_entsoe_TP.csv', encoding="utf-8", index=True)